# 03 · Random Forest Regressor 

**Bài toán:** dự đoán PM2.5 ở giờ kế tiếp (t+1) của cùng một trạm.

Notebook này triển khai Random Forest theo đúng quy trình dữ liệu của Notebook 02:
- Dùng lại `load_data`, `add_features`, `split_xy`.
- Chia Train/Validation và Test theo `target_datetime`.
- Dùng `cv_folds(..., n_splits=5)` với expanding-window TimeSeriesSplit.
- Tiền xử lý được fit bên trong từng fold để tránh leakage.
- Chỉ đánh giá trên 5 validation folds; **không dùng Test để lựa chọn mô hình**.

Phạm vi chỉ gồm Random Forest. Gradient Boosting và RandomizedSearchCV để dành cho giai đoạn sau.

## 1. Cấu hình và nạp thư viện

Tìm thư mục gốc dự án để notebook có thể chạy khi mở từ thư mục `notebooks/` hoặc từ root.

In [23]:
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "src").is_dir(), (
    "Không tìm thấy thư mục gốc dự án (thư mục chứa src/)."
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_data, time_split, cv_folds
from src.features import (
    FEATURE_COLUMNS,
    FFILL_LIMIT,
    PM25_LAGS,
    ROLLING_WINDOWS,
    add_features,
    build_preprocessor,
    split_xy,
)

RANDOM_STATE = 42
SAVE_ARTIFACTS = True

MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
EXPERIMENT_LOG = PROJECT_ROOT / "experiment_log.csv"

print("Project root:", PROJECT_ROOT)
print("scikit-learn:", sklearn.__version__)
print("pandas:", pd.__version__)

Project root: D:\github\sgu_finalproject_machinelearning_2026-2027
scikit-learn: 1.7.2
pandas: 2.3.3


## 2. Nạp dữ liệu và tạo đặc trưng

Đây là cùng quy trình trong Notebook 02. Target được tạo bởi `add_features()` từ PM2.5 của giờ kế tiếp trong cùng trạm. Các dòng thiếu target bị loại bỏ; target không được impute.

In [24]:
raw = load_data()
featured = add_features(raw)
labeled = featured.dropna(subset=["target"]).reset_index(drop=True)

print(f"Dữ liệu gốc: {raw.shape}")
print(f"Dữ liệu sau tạo feature: {featured.shape}")
print(f"Dữ liệu có target: {labeled.shape}")
print(f"Số trạm: {labeled['station'].nunique()}")
print(f"Số feature: {len(FEATURE_COLUMNS)}")

Dữ liệu gốc: (420768, 19)
Dữ liệu sau tạo feature: (420768, 35)
Dữ liệu có target: (412017, 35)
Số trạm: 12
Số feature: 27


## 3. Chia Train/Validation/Test và tạo đúng 5 folds

Dùng chính `time_split()` và `cv_folds()` của source gốc. Mốc Test là 01/09/2016 theo `src/data_loader.py`. Tất cả bản ghi cùng `target_datetime` được xếp cùng phía trong mỗi fold.

In [25]:
train_validation, test = time_split(labeled)
train_validation = train_validation.reset_index(drop=True)
test = test.reset_index(drop=True)

X_tv, y_tv = split_xy(train_validation)
folds = cv_folds(train_validation, n_splits=5)

fold_rows = []
for fold_id, (tr, va) in enumerate(folds, start=1):
    t_tr = train_validation.loc[tr, "target_datetime"]
    t_va = train_validation.loc[va, "target_datetime"]
    fold_rows.append({
        "fold": fold_id,
        "n_train": len(tr),
        "n_validation": len(va),
        "train_start": t_tr.min(),
        "train_end": t_tr.max(),
        "validation_start": t_va.min(),
        "validation_end": t_va.max(),
    })

fold_table = pd.DataFrame(fold_rows)
display(fold_table)

assert len(folds) == 5
for tr, va in folds:
    assert np.intersect1d(tr, va).size == 0
    assert (
        train_validation.loc[tr, "target_datetime"].max()
        < train_validation.loc[va, "target_datetime"].min()
    )

print(f"Train/Validation: {len(train_validation):,} rows")
print(f"Test: {len(test):,} rows (chưa sử dụng để đánh giá)")
print("Kiểm tra thứ tự thời gian và 5 folds: PASS")

,fold,n_train,n_validation,train_start,train_end,validation_start,validation_end
0,1,60258,60595,2013-03-01 01:00:00,2013-09-30 00:00:00,2013-09-30 01:00:00,2014-05-01 17:00:00
1,2,120853,59958,2013-03-01 01:00:00,2014-05-01 17:00:00,2014-05-01 18:00:00,2014-11-30 16:00:00
2,3,180811,59858,2013-03-01 01:00:00,2014-11-30 16:00:00,2014-11-30 17:00:00,2015-07-02 02:00:00
3,4,240669,60217,2013-03-01 01:00:00,2015-07-02 02:00:00,2015-07-02 03:00:00,2016-01-31 14:00:00
4,5,300886,60068,2013-03-01 01:00:00,2016-01-31 14:00:00,2016-01-31 15:00:00,2016-08-31 23:00:00


Train/Validation: 360,954 rows
Test: 51,063 rows (chưa sử dụng để đánh giá)
Kiểm tra thứ tự thời gian và 5 folds: PASS


## 4. Khởi tạo Random Forest Pipeline

Random Forest không cần StandardScaler. Tuy nhiên, vẫn cần median imputation, missing indicators và one-hot encoding. `build_preprocessor(scale=False)` thực hiện đúng các bước đó; transformer được fit riêng trên train của từng fold.

In [26]:
RF_PARAMS = {
    "n_estimators": 20,
    "max_features": 0.7,
    "max_depth": 16,
    "min_samples_leaf": 5,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

def make_random_forest():
    return Pipeline([
        ("prep", build_preprocessor(scale=False)),
        ("model", RandomForestRegressor(**RF_PARAMS)),
    ])

display(pd.Series(RF_PARAMS, name="value").to_frame())

,value
n_estimators,20.0
max_features,0.7
max_depth,16.0
min_samples_leaf,5.0
random_state,42.0
n_jobs,-1.0


## 5. Hàm tính metrics

In [27]:
def regression_metrics(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }

## 6. Huấn luyện và đánh giá 5-fold Cross Validation

Mỗi fold tạo một pipeline mới. Chỉ `X_train` của fold được dùng để fit tiền xử lý và Random Forest; validation chỉ dùng để dự đoán và tính metrics.

In [28]:
fold_results = []
cv_models = []
fold_runtime = []

for fold_id, (tr, va) in enumerate(folds, start=1):
    X_train, y_train = X_tv.iloc[tr], y_tv.iloc[tr]
    X_val, y_val = X_tv.iloc[va], y_tv.iloc[va]

    model = make_random_forest()
    start_time = time.perf_counter()
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    elapsed = time.perf_counter() - start_time

    metrics = regression_metrics(y_val, predictions)
    fold_results.append({
        "fold": fold_id,
        **metrics,
        "train_rows": len(tr),
        "validation_rows": len(va),
        "runtime_seconds": round(elapsed, 3),
    })
    cv_models.append(model)

    print(
        f"Fold {fold_id}/5 | "
        f"MAE={metrics['mae']:.4f} | "
        f"RMSE={metrics['rmse']:.4f} | "
        f"R2={metrics['r2']:.4f} | "
        f"{elapsed:.1f}s"
    )

rf_cv_metrics = pd.DataFrame(fold_results)

print("\nKết quả từng fold:")
display(rf_cv_metrics)


Fold 1/5 | MAE=14.9381 | RMSE=29.8863 | R2=0.8925 | 2.8s
Fold 2/5 | MAE=9.1974 | RMSE=16.0491 | R2=0.9493 | 11.8s
Fold 3/5 | MAE=10.1818 | RMSE=18.9940 | R2=0.9310 | 17.6s
Fold 4/5 | MAE=9.7558 | RMSE=18.6389 | R2=0.9596 | 24.1s
Fold 5/5 | MAE=8.5549 | RMSE=17.2853 | R2=0.9221 | 31.4s

Kết quả từng fold:


,fold,mae,rmse,r2,train_rows,validation_rows,runtime_seconds
0,1,14.938066,29.886266,0.892544,60258,60595,2.814
1,2,9.197387,16.049134,0.949266,120853,59958,11.771
2,3,10.181776,18.994013,0.931021,180811,59858,17.603
3,4,9.755823,18.638889,0.959637,240669,60217,24.080
4,5,8.554922,17.285319,0.922102,300886,60068,31.382


## 7. Tổng hợp kết quả Cross Validation

Báo cáo trung bình và độ lệch chuẩn giữa 5 folds. Các fold có kích thước validation khác nhau; bảng dưới đây trình bày trung bình không trọng số theo fold, tương tự cách tổng hợp trong Notebook 02.

In [29]:
metric_columns = ["mae", "rmse", "r2"]
rf_cv_summary = pd.DataFrame({
    "mean": rf_cv_metrics[metric_columns].mean(),
    "std": rf_cv_metrics[metric_columns].std(ddof=1),
})
display(rf_cv_summary)

print(f"Tổng thời gian CV: {rf_cv_metrics['runtime_seconds'].sum():.1f} giây")


,mean,std
mae,10.525595,2.541080
rmse,20.170724,5.555088
r2,0.930914,0.026037


Tổng thời gian CV: 87.7 giây


## 8. Kiểm tra kết quả Cross-Validation

Bảng dưới đây tổng hợp MAE, RMSE và R² của từng Fold. Phần đánh giá chỉ sử dụng các Fold validation; tập Test chưa được sử dụng.


In [30]:
print("Kết quả đánh giá theo từng fold:")
display(rf_cv_metrics[["fold", "mae", "rmse", "r2", "runtime_seconds"]])


Kết quả đánh giá theo từng fold:


,fold,mae,rmse,r2,runtime_seconds
0,1,14.938066,29.886266,0.892544,2.814
1,2,9.197387,16.049134,0.949266,11.771
2,3,10.181776,18.994013,0.931021,17.603
3,4,9.755823,18.638889,0.959637,24.080
4,5,8.554922,17.285319,0.922102,31.382


## 9. Fit mô hình cuối trên toàn bộ Train/Validation

Sau khi hoàn tất CV, fit một pipeline cuối trên toàn bộ Train/Validation để lưu dùng về sau. **Không đánh giá trên Test trong Ngày 2**; Test được giữ nguyên cho giai đoạn đánh giá cuối theo kế hoạch.

In [31]:
final_rf_model = make_random_forest()
final_rf_model.fit(X_tv, y_tv)

print("Đã fit mô hình cuối trên Train/Validation.")
print("Số dòng train/validation:", len(X_tv))
print("Test chưa được dự đoán hoặc dùng để chọn tham số.")

Đã fit mô hình cuối trên Train/Validation.
Số dòng train/validation: 360954
Test chưa được dự đoán hoặc dùng để chọn tham số.


## 10. Lưu mô hình, metrics và Experiment Log


In [32]:
DATASET_VERSION = (
    f"beijing_air_quality_standardized.csv; ffill<={FFILL_LIMIT}h; "
    f"lags={'/'.join(map(str, PM25_LAGS))}; "
    f"rolling={'/'.join(map(str, ROLLING_WINDOWS))}"
)

if SAVE_ARTIFACTS:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

    model_path = MODELS_DIR / "random_forest.joblib"
    metrics_path = OUTPUTS_DIR / "random_forest_cv_metrics.csv"
    metadata_path = MODELS_DIR / "random_forest_metadata.json"

    joblib.dump(final_rf_model, model_path, compress=3)
    rf_cv_metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")

    metadata = {
        "model": "RandomForestRegressor",
        "pipeline": "build_preprocessor(scale=False) + RandomForestRegressor",
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "sklearn_version": sklearn.__version__,
        "target": "PM2.5 at t+1 (same station)",
        "train_validation_rows": int(len(X_tv)),
        "test_start": "2016-09-01",
        "feature_columns": list(FEATURE_COLUMNS),
        "random_forest_params": RF_PARAMS,
        "cv": "TimeSeriesSplit expanding window, 5 folds, grouped by target_datetime",
        "cv_mean": {
            key: float(rf_cv_metrics[key].mean())
            for key in metric_columns
        },
        "cv_std": {
            key: float(rf_cv_metrics[key].std(ddof=1))
            for key in metric_columns
        },
        "test_evaluated": False,
        "dataset_version": DATASET_VERSION,
    }
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    log_columns = [
        "experiment_id", "model_name", "dataset_version",
        "mae", "rmse", "r2", "notes"
    ]
    log_row = pd.DataFrame([{
        "experiment_id": "random_forest_cv5",
        "model_name": "Random Forest Regressor",
        "dataset_version": DATASET_VERSION,
        "mae": round(float(rf_cv_metrics["mae"].mean()), 4),
        "rmse": round(float(rf_cv_metrics["rmse"].mean()), 4),
        "r2": round(float(rf_cv_metrics["r2"].mean()), 4),
        "notes": (
            "Mean of 5 validation folds; TimeSeriesSplit expanding window; "
            "same source split/features as Notebook 02; test not evaluated."
        ),
    }], columns=log_columns)

    if EXPERIMENT_LOG.exists() and EXPERIMENT_LOG.stat().st_size > 0:
        old_log = pd.read_csv(EXPERIMENT_LOG)
        old_log = old_log[
            old_log["experiment_id"] != "random_forest_cv5"
        ]
        updated_log = pd.concat([old_log, log_row], ignore_index=True)
    else:
        updated_log = log_row

    updated_log.to_csv(EXPERIMENT_LOG, index=False, encoding="utf-8-sig")

    print("Đã lưu:")
    for path in [model_path, metrics_path, metadata_path, EXPERIMENT_LOG]:
        print(" -", path.relative_to(PROJECT_ROOT))
else:
    print("SAVE_ARTIFACTS=False: không ghi artifact.")

Đã lưu:
 - models\random_forest.joblib
 - outputs\random_forest_cv_metrics.csv
 - models\random_forest_metadata.json
 - experiment_log.csv


## 11. Kiểm tra tải lại mô hình

Nạp lại file Joblib và xác nhận dự đoán trên một mẫu Train/Validation trùng với mô hình trong bộ nhớ. Đây chỉ là kiểm tra lưu/tải mô hình, không phải đánh giá Test.

In [33]:
if SAVE_ARTIFACTS:
    reloaded_model = joblib.load(MODELS_DIR / "random_forest.joblib")
    sample_X = X_tv.iloc[: min(1000, len(X_tv))]
    pred_original = final_rf_model.predict(sample_X)
    pred_reloaded = reloaded_model.predict(sample_X)

    assert np.allclose(pred_original, pred_reloaded)
    print("Nạp lại mô hình: PASS")
    print("Kiểu đối tượng:", type(reloaded_model))

Nạp lại mô hình: PASS
Kiểu đối tượng: <class 'sklearn.pipeline.Pipeline'>


## 12. Kết luận 

Sau khi chạy hết notebook, ghi nhận:
- MAE, RMSE, R² của từng fold và trung bình 5 folds.
- Thời gian chạy.
- Đường dẫn model và các file kết quả đã lưu.
- Những fold có sai số cao để phân tích sâu hơn.

**Chưa làm trong notebook :** Gradient Boosting, RandomizedSearchCV, tuning siêu tham số và đánh giá cuối cùng trên Test.